<a href="https://colab.research.google.com/github/thabomosenthal/symmetrical-fortnight/blob/main/Universal_API_Components.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Ep5 Code Lab: Free APIs & Integration
# Google Colab version

!pip -q install requests

import requests
import json
from google.colab import userdata

# 1. CONFIG
BASE_URL = "https://api.groq.com/openai/v1/chat/completions"
MODEL_NAME = "llama-3.3-70b-versatile"

# Read API key from Colab Secrets
# In Colab: click the key icon (Secrets) and create GROQ_API_KEY
API_KEY = userdata.get("GROQ_API_KEY")

if not API_KEY:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets.")

print("Connected. Ready.")

# 2. QUICK TEST — raw request
def raw_call(prompt, model=MODEL_NAME):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    response = requests.post(BASE_URL, headers=headers, json=payload)
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"]

print(raw_call("Say hello in one short sentence."))

Connected. Ready.
Hello.


In [ ]:
# 3. THE 5 UNIVERSAL API COMPONENTS

# 1. BASE URL — address of the model server
BASE_URL = "https://api.groq.com/openai/v1/chat/completions"

# 2. AUTH KEY — proves you are authorised
API_KEY = userdata.get("GROQ_API_KEY")

# 3. AUTH HEADER
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

# 4. MODEL STRING + 5. MESSAGES LIST
payload = {
    "model": "llama-3.3-70b-versatile",
    "messages": [
        {"role": "user", "content": "Hello"}
    ]
}

# Send and parse
response = requests.post(BASE_URL, headers=headers, json=payload)
response.raise_for_status()
data = response.json()

print(data["choices"][0]["message"]["content"])

Hello. How can I help you today?


In [ ]:
# 4. THE MESSAGES LIST IN DEPTH

full_call = requests.post(
    BASE_URL,
    headers=headers,
    json={
        "model": "llama-3.3-70b-versatile",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is the capital of France?"}
        ]
    }
)

full_call.raise_for_status()
data = full_call.json()

print("Answer:", data["choices"][0]["message"]["content"])
print("Tokens used:", data.get("usage", {}).get("total_tokens", "Not returned"))

Answer: The capital of France is Paris.
Tokens used: 56


In [ ]:
# 5. WRAPPER CLASS

class GroqClient:
    def __init__(self, api_key, base_url=BASE_URL, default_model=MODEL_NAME):
        self.api_key = api_key
        self.base_url = base_url
        self.default_model = default_model

    def chat(self, messages, model=None, temperature=0):
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        payload = {
            "model": model or self.default_model,
            "messages": messages,
            "temperature": temperature
        }

        response = requests.post(self.base_url, headers=headers, json=payload)
        response.raise_for_status()
        data = response.json()

        return data["choices"][0]["message"]["content"]

client = GroqClient(api_key=API_KEY)

In [ ]:
# 6. TESTING THE WRAPPER

# Single call
answer = client.chat([
    {"role": "user", "content": "What is 2 + 2?"}
])
print("Single call:", answer)

# Multi-turn conversation
conversation = [
    {"role": "user", "content": "My name is Alex."},
    {"role": "assistant", "content": "Hello Alex!"},
    {"role": "user", "content": "What is my name?"}
]

print()
print("Multi-turn:", client.chat(conversation))

Single call: 2 + 2 = 4.

Multi-turn: Your name is Alex.


In [ ]:
# 7. FIVE EPISODES INTO ONE PIPELINE

user_name = "Alex"

system_instruction = f"""
ROLE: You are a digital twin for {user_name}.
KNOWLEDGE: Activity data from Ep1, embeddings from Ep2, attention from Ep3, prompting strategy from Ep4.
CONSTRAINTS: Only recommend based on user data.
TONE: Direct and analytical.
"""

twin_messages = [
    {"role": "system", "content": system_instruction},
    {"role": "user", "content": "What should I do next?"}
]

response = client.chat(twin_messages)
print(response)

Analyzing your activity data from Ep1, I notice a pattern of engagement with creative tasks. Your embeddings from Ep2 indicate a strong interest in problem-solving and critical thinking. The attention data from Ep3 suggests you tend to focus on complex, challenging activities. 

Considering these insights, I recommend exploring a project that combines creative problem-solving with critical thinking. This could be a puzzle, a brain teaser, or a strategic game. The prompting strategy from Ep4 indicates you respond well to open-ended, thought-provoking questions. 

To proceed, I'll provide you with a prompt: "Design a innovative solution to a real-world problem." This should align with your interests and strengths, allowing you to leverage your creative and critical thinking skills. How would you like to approach this challenge?


In [ ]:
# 8. STRETCH: TWO-TURN CONVERSATION

two_turn_messages = [
    {"role": "system", "content": system_instruction},
    {"role": "user", "content": "What should I do next?"},
    {"role": "assistant", "content": client.chat(twin_messages)},
    {"role": "user", "content": "Give me the top 3 priorities only."}
]

print(client.chat(two_turn_messages))

Based on your data, my top 3 priorities for you are:

1. **Engage in creative problem-solving**: Leverage your creative skills to tackle complex challenges.
2. **Develop critical thinking**: Focus on activities that require critical thinking and analysis.
3. **Attempt a challenging puzzle**: Apply your problem-solving skills to a brain teaser or strategic game.
